# Hardware-Aware Model Architecture & Execution Mapper

End-to-end demo: profile **Qwen3-8B** on an NVIDIA H100 GPU and visualize across all 3 tabs.

- **Tab 1** — Workload Dashboard (per-request TTFT, decode, queue, GPU kernels)
- **Tab 2** — Request Inspector (nn.Module → op → kernel hierarchy with per-request variance)
- **Tab 3** — Architecture (one-time model structure with layer collapse)

All commands run on the **GPU host** unless noted otherwise.

## Prerequisites
# 🚀 vLLM Architecture & Profiling Map — Setup & Exploration
you can skip this step if you already have vllm installed and configure or if you want to 
follow this guide: https://docs.vllm.ai/en/latest/getting_started/quickstart/#nvidia-cuda
# 🚀 vLLM Architecture & Profiling Map — Setup & Exploration
### 1. Clone vllm repo
```bash
if [ ! -d "vllm" ]; then
  git clone https://github.com/vllm-project/vllm.git
fi
```

### 2. Create virtual environment
```bash
uv venv --python 3.12 --seed
source .venv/bin/activate
```

### 3. Install packages
```bash
uv pip install torch --torch-backend=auto 
uv pip install setuptools wheel setuptools-rust
uv pip install setuptools_scm ninja cmake numpy
cd vllm
MAX_JOBS=12 pip install -vv --no-build-isolation -e . # install it from source 
cd ../
```

### 4. System tools
- **nsys** (NVIDIA Nsight Systems) — comes with the CUDA toolkit (https://developer.nvidia.com/cuda-12-9-0-download-archive)
- Verify: `nsys --version` and `nvidia-smi`


## Tab 2: Per-Request Profiling (Request Inspector)

Single terminal, offline mode. Produces `per_request_profiles.json`.

`profile_serve.py` runs the model with `vllm.LLM` (offline, no server needed) and profiles each prompt individually using `torch.profiler`. Each prompt produces a Chrome trace (`.json.gz`) containing the full `nn.Module` → CPU op → GPU kernel hierarchy with real timing.

`profiler_to_diagram.py --per-request-dir` then reads these raw traces and builds a single merged module tree with per-request timing arrays. It correlates CPU ops to GPU kernels (via `External_id` → `cuda_runtime.correlation`, with timestamp-based fallback), collapses repeated decoder layers into a `×N` summary while preserving individual layer data for drill-down, and computes cross-layer variance stats.

**Why compress to JSON?** Raw Chrome traces are 50-100 MB each (171K+ events for Qwen3-8B). The processed `per_request_profiles.json` is ~1 MB — only the module tree structure, timing, and kernel mappings that the viewer needs. This makes SCP fast and the browser responsive.

In [1]:
#Set model
import os
#os.chdir("../../")
os.chdir("/home/redhat-et/lkesem/vllm_arch_map")
print("CURRENT DIRECTORY:", os.getcwd())

os.environ["MODEL"] = "Qwen/Qwen3-8B"

# Profile: accepts same dataset flags as vllm bench serve
#   --num-prompts: how many prompts to profile (default: 3)
#   --input-len / --output-len: token lengths
#   --dataset-name: random (default), sharegpt, sonnet, hf, custom
!python vllm_arch_map/profile_serve.py $MODEL \
    --num-prompts 3 --input-len 1024 --output-len 128

CURRENT DIRECTORY: /home/redhat-et/lkesem/vllm_arch_map
W1021 17:41:13.976000 813918 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1021 17:41:13.977000 813918 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1021 17:41:13.977000 813918 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=813918) INFO 10-21 17:41:16 [startup.py:96] non-default args: {'tokenizer': 'Qwen/Qwen3-8B', 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'profiler_config': ProfilerConfig(profiler='torch', torch_profiler_dir='/home/redhat-et/lkesem/vllm_arch_map/traces/per_request/profiles-c3c4f89d-ffd3-4b7d-978f-378d79a84a39', torch_profiler_activities=None, proton_profiler_dir='', proton_context='shadow', proton_data='tree', proton_backend=None, proton_mode=None, proton_hook=None, proton_output_format=None, proton_grap

In [2]:
# Verify traces were generated
!ls -lh traces/per_request/request_*.json*

-rw-rw-r-- 1 redhat-et redhat-et 4.2M Oct 20 23:34 traces/per_request/request_001.json.gz
-rw-rw-r-- 1 redhat-et redhat-et 4.2M Oct 20 23:34 traces/per_request/request_002.json.gz
-rw-rw-r-- 1 redhat-et redhat-et 4.2M Oct 20 23:34 traces/per_request/request_003.json.gz


In [3]:
# Process: build merged module tree with kernel correlation
!python vllm_arch_map/profiler_to_diagram.py \
    --per-request-dir traces/per_request --out_dir samples/

Processing 3 per-request traces from traces/per_request...

[Req-001] request_001.json.gz
Loading traces/per_request/request_001.json.gz (4.3 MB)...
  194031 events | 2032 modules | 15475 cpu_ops | 1922 kernels

[Req-002] request_002.json.gz
Loading traces/per_request/request_002.json.gz (4.3 MB)...
  193910 events | 2032 modules | 15475 cpu_ops | 1922 kernels

[Req-003] request_003.json.gz
Loading traces/per_request/request_003.json.gz (4.3 MB)...
  193879 events | 2032 modules | 15466 cpu_ops | 1922 kernels

Wrote samples/per_request_profiles.json (2.3 MB)
  3 requests merged into one tree


---
## Tab 1: Workload Dashboard

Two terminals needed. Produces `workload_data.json`.

`nvtx_serve.py` wraps vLLM's server with NVTX markers around each request, so nsys can segment the GPU kernel timeline per request. The benchmark client generates load, and Prometheus provides server-side aggregate metrics. `parse_workload.py` merges all three sources into one JSON.

In [5]:
# ┌─────────────────────────────────────────────────────────────┐
# │ TERMINAL 1: Start server under nsys profiler               │
# │ Run this in a separate terminal, not in the notebook.      │
# └─────────────────────────────────────────────────────────────┘
#
import os
os.environ["MODEL"] = "Qwen/Qwen3-8B"

import subprocess, os, time, urllib.request
os.makedirs("benchmark_outputs", exist_ok=True)

# Start server under nsys profiler in background
server = subprocess.Popen(
    ["nsys", "profile", "-t", "cuda,nvtx", "--cuda-memory-usage=true", "-o", "vllm_trace",
    "python", "vllm_arch_map/nvtx_serve.py", os.environ["MODEL"], 
    "--max-model-len", "8192", 
    "--gpu-memory-utilization", "0.85"],
    stdout=open("server_log.txt", "w"), stderr=subprocess.STDOUT
)

# Wait for server to be ready
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print(f"Server ready under nsys! (pid={server.pid})")
        break
    except:
        time.sleep(2)
        if i % 10 == 9:
            print(f"Still waiting... ({(i+1)*2}s)")


# Wait for: "Uvicorn running on http://0.0.0.0:8000"

print("Run the nsys+server command in Terminal 1, then continue.")

Still waiting... (20s)
Still waiting... (40s)
Still waiting... (60s)
Still waiting... (80s)
Still waiting... (100s)
Still waiting... (120s)
Still waiting... (140s)
Server ready under nsys! (pid=803225)
Run the nsys+server command in Terminal 1, then continue.


In [6]:
# ┌─────────────────────────────────────────────────────────────┐
# │ TERMINAL 2: Run benchmark + capture metrics                │
# └─────────────────────────────────────────────────────────────┘
import os
os.environ["MODEL"] = "Qwen/Qwen3-8B"

os.makedirs("benchmark_outputs", exist_ok=True)
!vllm bench serve --backend vllm --model $MODEL \
    --base-url http://localhost:8000 \
    --endpoint /v1/completions --dataset-name random \
    --num-prompts 3 --max-concurrency 1 \
    --random-input-len 1024 --random-output-len 128 \
    --save-result --save-detailed --result-dir benchmark_outputs/

W1020 22:30:49.026000 803740 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1020 22:30:49.027000 803740 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1020 22:30:49.027000 803740 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0xe00eb59e3b00>, trust_remote_code=False, seed=0, num_prompts=3, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_len

In [7]:
# Capture Prometheus metrics (server still running)
!curl -s http://localhost:8000/metrics > benchmark_outputs/prometheus.txt
print("Prometheus metrics saved. Now Ctrl+C the server in Terminal 1.")
server.terminate()
server.wait()
print("Server stopped. nsys wrote vllm_trace.nsys-rep")

Prometheus metrics saved. Now Ctrl+C the server in Terminal 1.
Server stopped. nsys wrote vllm_trace.nsys-rep


In [8]:
# After Ctrl+C: export nsys trace to sqlite
!nsys export --type=sqlite --output=./vllm_trace.sqlite vllm_trace.nsys-rep --force-overwrite true #nsys export --type=sqlite vllm_trace.nsys-rep --force-overwrite true

Processing 412462 events: [================================================100%]


In [ ]:
# Merge benchmark JSON + Prometheus + nsys sqlite
!ls benchmark_outputs

!python vllm_arch_map/parse_workload.py \
    --bench benchmark_outputs/*.json \
    --prometheus benchmark_outputs/prometheus.txt \
    --nsys vllm_trace.sqlite \
    --dataset-name random \
    -o samples/workload_data.json

prometheus.txt	vllm-infqps-concurrency1-Qwen3-8B-20261020-223123.json
Auto-detected 36 layers from Qwen/Qwen3-8B
Wrote samples/workload_data.json (20.9 KB)


---
## Tab 3: Architecture (one-time per model)

Single terminal. Produces `model_execution.json`.

`walk_model.py` walks the `nn.Module` tree and captures a PyTorch profiler trace. The trace is saved as a compressed `.json.gz` file (raw Chrome trace format). `profiler_to_diagram.py` reads it directly (no manual decompression needed) and builds the module hierarchy with kernel mappings.

In [10]:
# Walk model tree + capture profiler trace
!VLLM_ALLOW_INSECURE_SERIALIZATION=1 python vllm_arch_map/walk_model.py $MODEL \
    -o samples/model_tree.json --with-forward --profile-dir ./samples/traces/

W1020 22:33:21.339000 804065 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1020 22:33:21.339000 804065 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1020 22:33:21.339000 804065 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=804065) INFO 10-20 22:33:24 [startup.py:96] non-default args: {'max_model_len': 128, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enforce_eager': True, 'profiler_config': ProfilerConfig(profiler='torch', torch_profiler_dir='/home/redhat-et/lkesem/vllm_arch_map/samples/traces/profiles-9baeab6f-3be9-47d6-acb6-19c089c523e1', torch_profiler_activities=None, proton_profiler_dir='', proton_context='shadow', proton_data='tree', proton_backend=None, proton_mode=None, proton_hook=None, proton_output_format=None, proton_graph_attribution=False, torch_profiler_with_stack=True, torch_profiler_

In [11]:
# Process: build module tree with kernel correlation + layer collapse
!python vllm_arch_map/profiler_to_diagram.py \
    --out_dir samples/ --traces-dir samples/traces

Processing rank0.1789640479161371744.pt.trace.json...
Loading samples/traces/profiles/rank0.1789640479161371744.pt.trace.json (61.5 MB)...
  171176 events | 5080 modules | 40598 cpu_ops | 4575 kernels
Wrote samples/model_execution.json (2.7 MB)
  module_tree: Qwen3ForCausalLM | 36 layers | 30 kernel types


---
## View

### On the GPU host
```bash
python3 -m http.server 8765 --directory samples
```

### Or SCP to local machine
```bash
scp host:~/vllm_arch_map/samples/per_request_profiles.json samples/
scp host:~/vllm_arch_map/samples/workload_data.json samples/
scp host:~/vllm_arch_map/samples/model_execution.json samples/

python3 -m http.server 8765 --directory samples
open http://localhost:8765/model_viz.html
```

---
## Output Files

| File | Tab | Size | Contents |
|---|---|---|---|
| `per_request_profiles.json` | Tab 2 | ~1 MB | Merged module tree with per-request timing, kernel mapping, layer variance |
| `workload_data.json` | Tab 1 | ~50 KB | Per-request TTFT/decode/queue, GPU kernels, Prometheus metrics |
| `model_execution.json` | Tab 3 | ~300 KB | Module hierarchy, op→kernel correlation, layer collapse |

Tested on **DGX SPARK** (NVIDIA H100 80GB HBM3) with **Qwen/Qwen3-8B**.